# CIFAR T-Test

Run the pairwise T-test notebook after training a model. This notebook clones the repo into Colab if needed and runs the test logic directly in the notebook.

In [19]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
    print(f'Cloned repository to {repo_dir}')
else:
    print(f'Repository already present at {repo_dir}')

Repository already present at /content/DVBW


In [20]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / 'CIFAR', Path('/content/DVBW/CIFAR'), Path('/content/drive/MyDrive/DVBW/CIFAR')]
WORKDIR = next((path for path in candidates if (path / 'model.py').exists()), None)
if WORKDIR is None:
    raise FileNotFoundError('Could not find the CIFAR folder. Set WORKDIR manually if needed.')
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

Working directory: /content/DVBW/CIFAR


In [21]:
# Optional: mount Google Drive if your checkpoints are stored there.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; skipping Google Drive mount.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install -q scipy pillow numpy

In [23]:
from types import SimpleNamespace

args = SimpleNamespace(
    model='resnet',  # 'resnet' or 'vgg'
    model_dir='/content/drive/MyDrive/SPML_Models',
    model_filename='model_best_resnet_sinusoid_a0.05_freq4.pth',
    clean_model_filename='model_best_resnet_benign.pth.tar',
    target_label=0,
    num_img=100,
    test_batch=16,
    workers=2,
    gpu_id='0',
    margin=0.2,
    seed=666,
    primary_watermark_type='sinusoid',  # 'image', 'sinusoid', or 'benign'
    primary_trigger_path='./triggers/Trigger_cross.png',
    primary_alpha_path='./triggers/Alpha_cross.png',
    primary_sinusoid_frequency=4.0,
    primary_sinusoid_alpha=0.05,
    secondary_watermark_type='benign',  # 'image', 'sinusoid', or 'benign'
    secondary_trigger_path='./triggers/Trigger_line.png',
    secondary_alpha_path='./triggers/Alpha_line.png',
    secondary_sinusoid_frequency=4.0,
    secondary_sinusoid_alpha=0.05,
)
args


namespace(model='resnet',
          model_dir='/content/drive/MyDrive/SPML_Models',
          model_filename='model_best_resnet_sinusoid_a0.05_freq4.pth',
          clean_model_filename='model_best_resnet_benign.pth.tar',
          target_label=0,
          num_img=100,
          test_batch=16,
          workers=2,
          gpu_id='0',
          margin=0.2,
          seed=666,
          primary_watermark_type='sinusoid',
          primary_trigger_path='./triggers/Trigger_cross.png',
          primary_alpha_path='./triggers/Alpha_cross.png',
          primary_sinusoid_frequency=4.0,
          primary_sinusoid_alpha=0.05,
          secondary_watermark_type='benign',
          secondary_trigger_path='./triggers/Trigger_line.png',
          secondary_alpha_path='./triggers/Alpha_line.png',
          secondary_sinusoid_frequency=4.0,
          secondary_sinusoid_alpha=0.05)

In [24]:
import os
from pathlib import Path
import random
import shutil

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import ttest_rel

from model import *


class IdentityTransform(object):
    def __call__(self, img):
        return img.convert('RGB')


class ImageTriggerAppending(object):
    def __init__(self, trigger, alpha):
        self.trigger = np.array(trigger.clone().detach().permute(1, 2, 0) * 255)
        self.alpha = np.array(alpha.clone().detach().permute(1, 2, 0))

    def __call__(self, img):
        img_ = np.array(img).astype(np.float32)
        watermarked = (1 - self.alpha) * img_ + self.alpha * self.trigger
        return Image.fromarray(np.clip(watermarked, 0, 255).astype('uint8')).convert('RGB')


class SinusoidAppending(object):
    def __init__(self, frequency, alpha):
        self.frequency = float(frequency)
        self.alpha = float(alpha)

    def __call__(self, img):
        img_ = np.array(img).astype(np.float32)
        height, width, channels = img_.shape
        x = np.arange(width, dtype=np.float32)
        sinusoid = np.sin(2 * np.pi * self.frequency * x / width)
        sinusoid = ((sinusoid + 1.0) / 2.0) * 255.0
        sinusoid = np.tile(sinusoid, (height, 1))
        sinusoid = np.repeat(sinusoid[:, :, None], channels, axis=2)
        watermarked = (1 - self.alpha) * img_ + self.alpha * sinusoid
        return Image.fromarray(np.clip(watermarked, 0, 255).astype('uint8')).convert('RGB')


assert args.model in {'resnet', 'vgg'}
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_id
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
print(f'Running on device: {device}')

data_dir = WORKDIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
datasets.CIFAR10(root=str(data_dir), train=False, download=True)

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
if use_cuda:
    torch.cuda.manual_seed_all(args.seed)

print(f'Primary watermark type: {args.primary_watermark_type}')
print(f'Secondary watermark type: {args.secondary_watermark_type}')


def build_model(model_name):
    if model_name == 'resnet':
        return ResNet18()
    return vgg19_bn()


def build_watermark_operator(watermark_type, trigger_path=None, alpha_path=None, sinusoid_frequency=None, sinusoid_alpha=None):
    if watermark_type == 'benign':
        return IdentityTransform()
    if watermark_type == 'image':
        trigger = transforms.ToTensor()(Image.open(trigger_path))
        alpha = transforms.ToTensor()(Image.open(alpha_path))
        return ImageTriggerAppending(trigger=trigger, alpha=alpha)
    if watermark_type == 'sinusoid':
        return SinusoidAppending(frequency=sinusoid_frequency, alpha=sinusoid_alpha)
    raise ValueError("watermark_type must be 'image', 'sinusoid', or 'benign'.")


def build_dataset_transform(watermark_type, trigger_path=None, alpha_path=None, sinusoid_frequency=None, sinusoid_alpha=None):
    return transforms.Compose([
        build_watermark_operator(
            watermark_type=watermark_type,
            trigger_path=trigger_path,
            alpha_path=alpha_path,
            sinusoid_frequency=sinusoid_frequency,
            sinusoid_alpha=sinusoid_alpha,
        ),
        transforms.ToTensor(),
    ])


def collect_softmax_outputs(testloader, model):
    model.eval()
    outputs_all = []
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            outputs_all += torch.nn.functional.softmax(outputs, dim=1).cpu().numpy().tolist()
    return np.array(outputs_all)


def select_non_target_examples(dataset, target_label, num_img):
    candidate_indices = [idx for idx, label in enumerate(dataset.targets) if label != target_label]
    random.shuffle(candidate_indices)
    selected_indices = candidate_indices[:num_img]
    selected_img = [dataset.data[idx] for idx in selected_indices]
    selected_target = [dataset.targets[idx] for idx in selected_indices]
    return selected_indices, selected_img, selected_target


def attach_subset(dataset, selected_img, selected_target):
    dataset.data = selected_img
    dataset.targets = selected_target
    return dataset


def normalize_state_dict_for_model(state_dict, model):
    model_keys = list(model.state_dict().keys())
    state_keys = list(state_dict.keys())
    if not state_keys:
        return state_dict
    model_uses_module = model_keys[0].startswith('module.')
    state_uses_module = state_keys[0].startswith('module.')
    if model_uses_module == state_uses_module:
        return state_dict
    if state_uses_module:
        return {k.replace('module.', '', 1): v for k, v in state_dict.items()}
    return {f'module.{k}': v for k, v in state_dict.items()}


def maybe_mount_google_drive(path):
    path = Path(path)
    if '/content/drive' not in str(path):
        return
    if path.exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Could not mount Google Drive automatically: {exc}')


def resolve_model_paths():
    model_dir = Path(args.model_dir)
    maybe_mount_google_drive(model_dir)
    model_path = model_dir / args.model_filename
    clean_model_path = model_dir / args.clean_model_filename
    if not model_path.exists():
        raise FileNotFoundError(f'Model checkpoint not found: {model_path}')
    if not clean_model_path.exists():
        raise FileNotFoundError(f'Clean model checkpoint not found: {clean_model_path}')
    return model_path, clean_model_path


def stage_checkpoint_locally(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if '/content/drive' not in str(checkpoint_path):
        return checkpoint_path
    cache_dir = Path('/tmp/spml_models_cache')
    cache_dir.mkdir(parents=True, exist_ok=True)
    staged_path = cache_dir / checkpoint_path.name
    try:
        shutil.copy2(checkpoint_path, staged_path)
    except OSError as exc:
        print(f'Checkpoint copy from Google Drive failed once: {exc}')
        maybe_mount_google_drive(checkpoint_path)
        shutil.copy2(checkpoint_path, staged_path)
    return staged_path


def safe_torch_load(checkpoint_path, map_location):
    checkpoint_path = Path(checkpoint_path)
    staged_path = stage_checkpoint_locally(checkpoint_path)
    try:
        return torch.load(staged_path, map_location=map_location)
    except OSError as exc:
        if '/content/drive' not in str(checkpoint_path):
            raise
        print(f'torch.load failed from staged checkpoint: {exc}')
        maybe_mount_google_drive(checkpoint_path)
        staged_path = stage_checkpoint_locally(checkpoint_path)
        return torch.load(staged_path, map_location=map_location)


def main():
    model_path, clean_model_path = resolve_model_paths()
    print(f'Using model: {model_path}')
    print(f'Using clean baseline: {clean_model_path}')

    main_model = build_model(args.model)
    clean_model = build_model(args.model)

    main_checkpoint = safe_torch_load(model_path, map_location=device)
    clean_checkpoint = safe_torch_load(clean_model_path, map_location=device)

    if use_cuda:
        main_model = torch.nn.DataParallel(main_model).to(device)
        clean_model = torch.nn.DataParallel(clean_model).to(device)
    else:
        main_model = main_model.to(device)
        clean_model = clean_model.to(device)
    main_state_dict = normalize_state_dict_for_model(main_checkpoint['state_dict'], main_model)
    clean_state_dict = normalize_state_dict_for_model(clean_checkpoint['state_dict'], clean_model)
    main_model.load_state_dict(main_state_dict)
    clean_model.load_state_dict(clean_state_dict)
    main_model.eval()
    clean_model.eval()
    if use_cuda:
        cudnn.benchmark = True

    transform_primary = build_dataset_transform(
        watermark_type=args.primary_watermark_type,
        trigger_path=args.primary_trigger_path,
        alpha_path=args.primary_alpha_path,
        sinusoid_frequency=args.primary_sinusoid_frequency,
        sinusoid_alpha=args.primary_sinusoid_alpha,
    )
    transform_secondary = build_dataset_transform(
        watermark_type=args.secondary_watermark_type,
        trigger_path=args.secondary_trigger_path,
        alpha_path=args.secondary_alpha_path,
        sinusoid_frequency=args.secondary_sinusoid_frequency,
        sinusoid_alpha=args.secondary_sinusoid_alpha,
    )
    transform_benign = transforms.Compose([transforms.ToTensor()])

    dataloader = datasets.CIFAR10
    test_set_basic = dataloader(root=str(data_dir), train=False, download=True)
    testset_primary = dataloader(root=str(data_dir), train=False, download=True, transform=transform_primary)
    testset_secondary = dataloader(root=str(data_dir), train=False, download=True, transform=transform_secondary)
    testset_benign = dataloader(root=str(data_dir), train=False, download=True, transform=transform_benign)

    image_idx, selected_img, selected_target = select_non_target_examples(test_set_basic, args.target_label, args.num_img)

    testset_primary = attach_subset(testset_primary, selected_img, selected_target)
    testset_secondary = attach_subset(testset_secondary, selected_img, selected_target)
    testset_benign = attach_subset(testset_benign, selected_img, selected_target)

    primary_loader = torch.utils.data.DataLoader(testset_primary, batch_size=args.test_batch, shuffle=False, num_workers=args.workers)
    secondary_loader = torch.utils.data.DataLoader(testset_secondary, batch_size=args.test_batch, shuffle=False, num_workers=args.workers)
    benign_loader = torch.utils.data.DataLoader(testset_benign, batch_size=args.test_batch, shuffle=False, num_workers=args.workers)

    output_main_primary = collect_softmax_outputs(primary_loader, main_model)
    output_main_benign = collect_softmax_outputs(benign_loader, main_model)
    output_clean_primary = collect_softmax_outputs(primary_loader, clean_model)
    output_clean_benign = collect_softmax_outputs(benign_loader, clean_model)

    p_main_primary = output_main_primary[:, args.target_label]
    p_main_benign = output_main_benign[:, args.target_label]
    p_clean_primary = output_clean_primary[:, args.target_label]
    p_clean_benign = output_clean_benign[:, args.target_label]

    t_primary = ttest_rel(p_main_benign + args.margin, p_main_primary, alternative='less')
    t_model_independent = ttest_rel(p_clean_benign + args.margin, p_clean_primary, alternative='less')

    secondary_enabled = args.secondary_watermark_type != 'benign'
    if secondary_enabled:
        output_main_secondary = collect_softmax_outputs(secondary_loader, main_model)
        p_main_secondary = output_main_secondary[:, args.target_label]
        t_secondary = ttest_rel(p_main_benign + args.margin, p_main_secondary, alternative='less')
    else:
        p_main_secondary = None
        t_secondary = None

    path_folder = model_path.parent
    output_path = path_folder / f"Ttest_{args.primary_watermark_type}_{args.secondary_watermark_type}_{args.num_img}.txt"

    print(
        f"Primary Ttest ({args.primary_watermark_type} vs benign) p-value: {t_primary[1]:.4e}, "
        f"average delta P: {np.mean(p_main_primary - p_main_benign):.4e}"
    )
    print(
        f"Model Independent Ttest on {args.primary_watermark_type} p-value: {t_model_independent[1]:.4e}, "
        f"average delta P: {np.mean(p_clean_primary - p_clean_benign):.4e}"
    )
    if secondary_enabled:
        print(
            f"Secondary Ttest ({args.secondary_watermark_type} vs benign) p-value: {t_secondary[1]:.4e}, "
            f"average delta P: {np.mean(p_main_secondary - p_main_benign):.4e}"
        )
    else:
        print('Secondary Ttest skipped because secondary_watermark_type is benign.')

    with open(output_path, 'w') as f:
        f.write(f"model_path={model_path}\n")
        f.write(f"clean_model_path={clean_model_path}\n")
        f.write(f"primary_watermark_type={args.primary_watermark_type}\n")
        f.write(f"secondary_watermark_type={args.secondary_watermark_type}\n")
        header = 'image_idx main_primary main_benign clean_primary clean_benign'
        if secondary_enabled:
            header += ' main_secondary'
        f.write(header + '\n')
        for i in range(len(p_main_primary)):
            row = [
                str(image_idx[i]),
                f'{p_main_primary[i]:.4e}',
                f'{p_main_benign[i]:.4e}',
                f'{p_clean_primary[i]:.4e}',
                f'{p_clean_benign[i]:.4e}',
            ]
            if secondary_enabled:
                row.append(f'{p_main_secondary[i]:.4e}')
            f.write(' '.join(row) + '\n')
        f.write(
            f"Primary Ttest ({args.primary_watermark_type} vs benign) p-value: {t_primary[1]:.4e}, "
            f"average delta P: {np.mean(p_main_primary - p_main_benign):.4e}\n"
        )
        f.write(
            f"Model Independent Ttest on {args.primary_watermark_type} p-value: {t_model_independent[1]:.4e}, "
            f"average delta P: {np.mean(p_clean_primary - p_clean_benign):.4e}\n"
        )
        if secondary_enabled:
            f.write(
                f"Secondary Ttest ({args.secondary_watermark_type} vs benign) p-value: {t_secondary[1]:.4e}, "
                f"average delta P: {np.mean(p_main_secondary - p_main_benign):.4e}\n"
            )
        else:
            f.write('Secondary Ttest skipped because secondary_watermark_type is benign.\n')
    print(f'Saved results to {output_path}')


main()


Running on device: cuda
Primary watermark type: sinusoid
Secondary watermark type: benign
Using model: /content/drive/MyDrive/SPML_Models/model_best_resnet_sinusoid_a0.05_freq4.pth
Using clean baseline: /content/drive/MyDrive/SPML_Models/model_best_resnet_benign.pth.tar
Primary Ttest (sinusoid vs benign) p-value: 1.0236e-84, average delta P: 9.7148e-01
Model Independent Ttest on sinusoid p-value: 1.0000e+00, average delta P: -4.9873e-04
Secondary Ttest skipped because secondary_watermark_type is benign.
Saved results to /content/drive/MyDrive/SPML_Models/Ttest_sinusoid_benign_100.txt
